# SimCLR Features for Football Images

This tutorial trains a YOLO backbone with SimCLR on the football player detection dataset. The object labels are not used during training. They are read later only to color the t-SNE visualization.

**Audience**

Students who know basic Python and have seen a PyTorch training loop.

**Learning goals**

- Prepare the Kaggle football dataset for label-free learning
- Train SimCLR on two NVIDIA T4 GPUs
- Save and reload the learned YOLO backbone
- Extract image-level SSL features
- Visualize the feature space with t-SNE
- Inspect nearest neighbors in the learned representation space


## How SimCLR learns

For every image, SimCLR creates two independently augmented views. The encoder and projection head map the views to normalized vectors. Their cosine similarity is $s_{i,j}=z_i^Tz_j$.

For a positive pair $(i,j)$, the NT-Xent loss is $\ell_{i,j}=-\log\frac{\exp(s_{i,j}/\tau)}{\sum_{k\neq i}\exp(s_{i,k}/\tau)}$, where $\tau$ is the temperature. The positive pair is pulled together while the other images in the batch act as negatives.

The projection head is used to optimize the contrastive loss. The t-SNE section uses the backbone output before the projection head because that is the representation transferred to downstream tasks.


## Notebook roadmap

1. Configure the Kaggle runtime
2. Find and inspect the dataset
3. Configure SimCLR
4. Inspect the two augmented views
5. Train on T4 x2
6. Review the training loss
7. Extract validation features
8. Create the t-SNE plot
9. Inspect nearest neighbors
10. Try an exercise


## 1. Configure the Kaggle runtime

Open **Notebook options**, select **GPU T4 x2**, and enable Internet access. Add the `iasadpanwhar/football-player-detection-yolov8` dataset through **Add Input**. The notebook reads the mounted files directly and does not call the Kaggle competition API.


In [ ]:
%pip install -q --upgrade "git+https://github.com/rifat963/ssl-detection-lab.git@main" scikit-learn seaborn


In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import replace
import json
import random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from packaging.version import Version
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO

import ssldet
from ssldet import PretrainConfig, available_ssl_modules, launch_distributed_pretrain
from ssldet.backbones import YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")

assert Version(ssldet.__version__) >= Version("0.8.1")
assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."

GPU_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda:0")
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]

pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "CUDA": torch.version.cuda,
    "GPU count": GPU_COUNT,
    "GPUs": ", ".join(GPU_NAMES),
    "SSL modules": ", ".join(available_ssl_modules()),
})


A T4 x2 session should report two GPUs. Training still runs with another GPU count, but the batch calculation and runtime will differ.


## 2. Find and inspect the dataset

The first path below matches the full path supplied with the dataset. The second path covers Kaggle's shorter mounted-input layout.


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]

DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if DATASET_ROOT is None:
    mounted_inputs = sorted(str(path) for path in Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        "Attach the football-player-detection-yolov8 dataset with Add Input. "
        f"Mounted inputs: {mounted_inputs}"
    )

SPLIT_PATHS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

dataset_summary = []
for split, paths in SPLIT_PATHS.items():
    images = image_files(paths["images"])
    labels = sorted(paths["labels"].glob("*.txt"))
    dataset_summary.append({"split": split, "images": len(images), "labels": len(labels)})

pd.DataFrame(dataset_summary).set_index("split")


SimCLR uses `train/images` only. No annotation path is passed to the trainer. The validation images and labels are kept separate for representation analysis.


In [ ]:
train_images = image_files(SPLIT_PATHS["train"]["images"])
sample_count = min(8, len(train_images))
sample_paths = random.Random(SEED).sample(train_images, sample_count)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, path in zip(axes.flat, sample_paths):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name, fontsize=9)
plt.suptitle("Training images", fontsize=16)
plt.tight_layout()
plt.show()


## 3. Configure SimCLR

`yolo26n.yaml` initializes the architecture without COCO weights, giving a strict label-free pretraining run. Change it to `yolo26n.pt` if the experiment should start from supervised COCO weights and then adapt to football images.

`batch_size` is the number of source images per GPU. Each GPU creates two views of every source image. Gradient accumulation can increase the optimizer batch, but it does not add more simultaneous contrastive negatives.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/simclr_football")
EPOCHS = 10
PER_GPU_BATCH_SIZE = 32

config = PretrainConfig(
    method="simclr",
    image_roots=[str(SPLIT_PATHS["train"]["images"])],
    output_dir=str(OUTPUT_DIR),
    yolo_model="yolo26n.yaml",
    epochs=EPOCHS,
    batch_size=PER_GPU_BATCH_SIZE,
    image_size=224,
    workers=2,
    max_images=None,
    seed=SEED,
    learning_rate=3e-4,
    min_learning_rate=3e-6,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_accum_steps=1,
    gradient_clip=5.0,
    amp=True,
    projection_dim=128,
    hidden_dim=512,
    temperature=0.20,
    save_every=1,
).validate()

pd.Series({
    "model": config.yolo_model,
    "epochs": config.epochs,
    "training images": len(train_images),
    "batch per GPU": config.batch_size,
    "source images per DDP step": config.batch_size * GPU_COUNT,
    "augmented views per DDP step": 2 * config.batch_size * GPU_COUNT,
    "image size": config.image_size,
    "temperature": config.temperature,
    "mixed precision": config.amp,
})


## 4. Inspect the two augmented views

The library applies random resized cropping, horizontal flipping, color jitter, grayscale conversion, blur, tensor conversion, and ImageNet normalization. Both views come from the same source image but receive independent random transformations.


In [ ]:
preview_dataset = UnlabeledImageDataset(train_images, build_transform(config))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def display_tensor(tensor):
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(4, 2, figsize=(8, 15))
for row, index in enumerate(range(4)):
    first_view, second_view = preview_dataset[index]
    axes[row, 0].imshow(display_tensor(first_view))
    axes[row, 1].imshow(display_tensor(second_view))
    axes[row, 0].set_ylabel(f"Image {index + 1}")
    for axis in axes[row]:
        axis.set_xticks([])
        axis.set_yticks([])
axes[0, 0].set_title("View A")
axes[0, 1].set_title("View B")
plt.tight_layout()
plt.show()


The scene content should remain recognizable across each pair. If the crop removes almost every football object too often, increase the minimum crop scale in the library transform.


## 5. Train on T4 x2

The launcher writes the configuration to the output directory and starts one distributed process per visible GPU. The progress bar reports the running loss and learning rate. Checkpoints, history, and the updated YOLO model are written under `/kaggle/working/simclr_football`.


In [ ]:
training_result = launch_distributed_pretrain(
    config,
    num_processes=GPU_COUNT,
    config_path=OUTPUT_DIR / "simclr_config.yaml",
    check=True,
)

pd.Series({
    "completed": training_result.succeeded,
    "seconds": round(training_result.seconds, 1),
    "output directory": str(training_result.output_dir),
    "configuration": str(training_result.config_path),
})


A decreasing loss shows that the model is learning to identify positive augmented pairs relative to other images in the batch. The absolute loss value should not be interpreted as detection accuracy.


## 6. Review the training loss


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
display(history.round(6))

fig, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", linewidth=2.5, ax=axis)
axis.set_title("SimCLR training loss")
axis.set_xlabel("Epoch")
axis.set_ylabel("NT-Xent loss")
axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()


In [ ]:
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
YOLO_CHECKPOINT = Path(manifest["outputs"]["yolo_checkpoint"])
SSL_CHECKPOINT = Path(manifest["outputs"]["ssl_checkpoint"])

pd.Series({
    "initialization": manifest["initialization"],
    "unlabeled images": manifest["unlabeled_images"],
    "world size": manifest["world_size"],
    "best loss": manifest["best_loss"],
    "YOLO checkpoint": str(YOLO_CHECKPOINT),
    "full SSL checkpoint": str(SSL_CHECKPOINT),
})


The YOLO checkpoint contains the learned backbone in a detector-compatible model. The full SSL checkpoint also contains the projection head, optimizer, scheduler, scaler, history, and training configuration.


## 7. Extract validation features

The validation images were not used for SSL optimization. Each image is transformed deterministically, passed through the trained backbone, globally pooled, and normalized.

An object-detection image can contain several classes. For visualization, each image receives the rarest class present in that image. This prevents the common player class from hiding every rarer football role. This image-level label is only a plotting aid, not a training target.


In [ ]:
yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
dataset_yaml = yaml_candidates[0] if yaml_candidates else None
metadata = yaml.safe_load(dataset_yaml.read_text()) if dataset_yaml else {}
raw_names = metadata.get("names", {})

if isinstance(raw_names, list):
    CLASS_NAMES = {index: name for index, name in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {int(index): name for index, name in raw_names.items()}
else:
    CLASS_NAMES = {}

def object_classes(label_path):
    if not label_path.exists():
        return []
    rows = [line.split() for line in label_path.read_text().splitlines() if line.strip()]
    return [int(float(row[0])) for row in rows]

validation_images = image_files(SPLIT_PATHS["valid"]["images"])
validation_class_lists = [
    object_classes(SPLIT_PATHS["valid"]["labels"] / f"{path.stem}.txt")
    for path in validation_images
]
class_frequency = Counter(class_id for values in validation_class_lists for class_id in values)

if not CLASS_NAMES:
    CLASS_NAMES = {class_id: f"class {class_id}" for class_id in sorted(class_frequency)}

class_table = pd.DataFrame([
    {"class id": class_id, "class name": CLASS_NAMES.get(class_id, f"class {class_id}"), "objects": count}
    for class_id, count in sorted(class_frequency.items())
]).set_index("class id")
class_table


In [ ]:
EMBEDDING_LIMIT = 500
selected_paths = validation_images
if len(selected_paths) > EMBEDDING_LIMIT:
    selected_paths = sorted(random.Random(SEED).sample(selected_paths, EMBEDDING_LIMIT))

evaluation_transform = v2.Compose([
    v2.Resize(config.image_size + 32, antialias=True),
    v2.CenterCrop(config.image_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def image_label(path):
    label_path = SPLIT_PATHS["valid"]["labels"] / f"{path.stem}.txt"
    values = set(object_classes(label_path))
    return min(values, key=lambda class_id: class_frequency[class_id]) if values else -1

class FootballFeatureDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, image_label(path), path.name

feature_dataset = FootballFeatureDataset(selected_paths, evaluation_transform)
feature_loader = DataLoader(
    feature_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

len(feature_dataset)


In [ ]:
detector = YOLO(str(YOLO_CHECKPOINT))
encoder = YOLOBackboneEncoder(detector.model).to(DEVICE).eval()

feature_batches = []
label_batches = []
file_names = []

with torch.inference_mode():
    for images, labels, names in tqdm(feature_loader, desc="Extracting SSL features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(F.normalize(features, dim=1).cpu())
        label_batches.append(labels.numpy())
        file_names.extend(names)

feature_matrix = torch.cat(feature_batches)
label_ids = np.concatenate(label_batches)

pd.Series({
    "images": feature_matrix.shape[0],
    "feature dimensions": feature_matrix.shape[1],
    "feature norm mean": feature_matrix.norm(dim=1).mean().item(),
})


## 8. Create the t-SNE plot

t-SNE converts the high-dimensional feature vectors into two dimensions while trying to preserve local neighborhoods. Nearby points are more informative than the absolute axis values. Distances between far-away clusters should not be treated as calibrated measurements.


In [ ]:
sample_total = len(feature_matrix)
perplexity = min(30.0, max(2.0, (sample_total - 1) / 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=SEED,
)
coordinates = tsne.fit_transform(feature_matrix.numpy())

plot_frame = pd.DataFrame({
    "t-SNE 1": coordinates[:, 0],
    "t-SNE 2": coordinates[:, 1],
    "class": [CLASS_NAMES.get(int(class_id), "unlabelled") for class_id in label_ids],
    "image": file_names,
})

fig, axis = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=plot_frame,
    x="t-SNE 1",
    y="t-SNE 2",
    hue="class",
    palette="tab10",
    s=65,
    alpha=0.82,
    edgecolor="white",
    linewidth=0.35,
    ax=axis,
)
axis.set_title("t-SNE of SimCLR football features", fontsize=16)
axis.legend(title="Rarest object present", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


A useful representation often places visually or semantically related scenes near one another. Perfect class separation is not expected because SimCLR never saw the labels and each image can contain several object classes. Run t-SNE with the same seed when comparing experiments.


## 9. Inspect nearest neighbors

Cosine similarity in the original feature space provides a more direct check than the two-dimensional t-SNE projection. The first image is the query; the remaining images are its closest neighbors.


In [ ]:
QUERY_INDEX = 0
NEIGHBOR_COUNT = min(5, len(feature_matrix) - 1)
similarities = feature_matrix @ feature_matrix[QUERY_INDEX]
neighbor_indices = torch.topk(similarities, k=NEIGHBOR_COUNT + 1).indices.tolist()
neighbor_indices = [index for index in neighbor_indices if index != QUERY_INDEX][:NEIGHBOR_COUNT]
display_indices = [QUERY_INDEX] + neighbor_indices

fig, axes = plt.subplots(1, len(display_indices), figsize=(4 * len(display_indices), 4))
for position, (axis, index) in enumerate(zip(axes, display_indices)):
    with Image.open(selected_paths[index]) as image:
        axis.imshow(image.convert("RGB"))
    class_name = CLASS_NAMES.get(int(label_ids[index]), "unlabelled")
    title = "Query" if position == 0 else f"Similarity {similarities[index]:.3f}"
    axis.set_title(f"{title}\n{class_name}")
    axis.axis("off")
plt.tight_layout()
plt.show()


## 10. Exercise

Train a shorter experiment with temperature $\tau=0.10$. Compare its loss curve, t-SNE neighborhoods, and nearest neighbors with the original run. Keep the random seed and selected validation images unchanged so that the comparison is fair.

Before running it, predict whether a lower temperature will make the contrastive softmax sharper or smoother.


In [ ]:
exercise_config = replace(
    config,
    temperature=0.10,
    epochs=5,
    output_dir="/kaggle/working/simclr_football_temperature_010",
).validate()

pd.Series({
    "temperature": exercise_config.temperature,
    "epochs": exercise_config.epochs,
    "output directory": exercise_config.output_dir,
})


Run the exercise with `launch_distributed_pretrain(exercise_config, num_processes=GPU_COUNT)`, then repeat the feature extraction and visualization cells with its YOLO checkpoint. A lower temperature makes the similarity softmax sharper and increases the emphasis on hard negatives.


## Practical checks

- An out-of-memory error usually means the per-GPU batch is too large. Reduce it from 32 to 16.
- Missing dataset paths mean the dataset was not attached through Add Input.
- A version assertion failure means the GitHub branch does not yet contain version 0.5.0.
- A falling SSL loss does not prove that object detection improved. Fine-tune the saved YOLO checkpoint on the training labels and evaluate it on the test split for that conclusion.
- t-SNE is exploratory. Use a linear probe, k-nearest-neighbor accuracy, or downstream detection metrics for quantitative comparisons.
